In [1]:
import os
import math
import numpy as np
import pandas as pd

from collections import Counter, defaultdict
import conllu

In [2]:
def load_conllu(file_path):

    with open(file_path, "r", encoding="utf-8") as f:
        data = f.read()

    sentences = conllu.parse(data)

    return sentences

In [3]:
train_path = "dataset/en_ewt-ud-train.conllu"
test_path = "dataset/en_ewt-ud-test.conllu"

train_data = load_conllu(train_path)
test_data = load_conllu(test_path)

print("Training sentences:", len(train_data))
print("Testing sentences:", len(test_data))

Training sentences: 12544
Testing sentences: 2077


In [4]:
def extract_sentences(data):

    sentences = []

    for sentence in data:

        words = []
        tags = []

        for token in sentence:

            if isinstance(token["id"], int):

                words.append(token["form"].lower())
                tags.append(token["upos"])

        sentences.append((words, tags))

    return sentences

In [5]:
train_sentences = extract_sentences(train_data)
test_sentences = extract_sentences(test_data)

print(train_sentences[0])

(['al', '-', 'zaman', ':', 'american', 'forces', 'killed', 'shaikh', 'abdullah', 'al', '-', 'ani', ',', 'the', 'preacher', 'at', 'the', 'mosque', 'in', 'the', 'town', 'of', 'qaim', ',', 'near', 'the', 'syrian', 'border', '.'], ['PROPN', 'PUNCT', 'PROPN', 'PUNCT', 'ADJ', 'NOUN', 'VERB', 'PROPN', 'PROPN', 'PROPN', 'PUNCT', 'PROPN', 'PUNCT', 'DET', 'NOUN', 'ADP', 'DET', 'NOUN', 'ADP', 'DET', 'NOUN', 'ADP', 'PROPN', 'PUNCT', 'ADP', 'DET', 'ADJ', 'NOUN', 'PUNCT'])


In [6]:
transition_counts = defaultdict(Counter)
tag_counts = Counter()

for words, tags in train_sentences:

    previous_tag = "<START>"

    for tag in tags:

        transition_counts[previous_tag][tag] += 1
        tag_counts[previous_tag] += 1

        previous_tag = tag

    transition_counts[previous_tag]["<END>"] += 1

In [7]:
transition_probs = defaultdict(dict)

for previous_tag in transition_counts:

    total = sum(
        transition_counts[previous_tag].values()
    )

    for current_tag in transition_counts[previous_tag]:

        transition_probs[previous_tag][current_tag] = (
            transition_counts[previous_tag][current_tag]
            / total
        )

In [8]:
emission_counts = defaultdict(Counter)

for words, tags in train_sentences:

    for word, tag in zip(words, tags):

        emission_counts[tag][word] += 1

In [9]:
emission_probs = defaultdict(dict)

for tag in emission_counts:

    total = sum(
        emission_counts[tag].values()
    )

    for word in emission_counts[tag]:

        emission_probs[tag][word] = (
            emission_counts[tag][word]
            / total
        )

In [10]:
tags = list(tag_counts.keys())

tags.remove("<START>")

print("POS Tags:")
print(tags)

POS Tags:
['PROPN', 'PUNCT', 'ADJ', 'NOUN', 'VERB', 'DET', 'ADP', 'AUX', 'PRON', 'PART', 'SCONJ', 'NUM', 'ADV', 'CCONJ', 'INTJ', 'X', 'SYM']


In [11]:
def viterbi(words):

    viterbi_table = []
    backpointer = []

    # First word
    first_word = words[0]

    current_probs = {}
    current_backpointer = {}

    for tag in tags:

        transition = transition_probs["<START>"].get(tag, 0)

        emission = emission_probs[tag].get(first_word, 0)

        if transition > 0 and emission > 0:

            current_probs[tag] = (
                math.log(transition)
                + math.log(emission)
            )

        else:

            current_probs[tag] = float("-inf")

        current_backpointer[tag] = None

    viterbi_table.append(current_probs)
    backpointer.append(current_backpointer)

    # Remaining words
    for i in range(1, len(words)):

        word = words[i]

        current_probs = {}
        current_backpointer = {}

        for current_tag in tags:

            emission = emission_probs[
                current_tag
            ].get(word, 0)

            if emission == 0:

                current_probs[current_tag] = float("-inf")
                current_backpointer[current_tag] = None
                continue

            best_score = float("-inf")
            best_previous = None

            for previous_tag in tags:

                transition = transition_probs[
                    previous_tag
                ].get(current_tag, 0)

                previous_score = viterbi_table[i - 1].get(
                    previous_tag,
                    float("-inf")
                )

                if transition == 0 or previous_score == float("-inf"):
                    continue

                score = (
                    previous_score
                    + math.log(transition)
                    + math.log(emission)
                )

                if score > best_score:

                    best_score = score
                    best_previous = previous_tag

            current_probs[current_tag] = best_score
            current_backpointer[current_tag] = best_previous

        viterbi_table.append(current_probs)
        backpointer.append(current_backpointer)

    # Find best final tag
    best_final_tag = max(
        viterbi_table[-1],
        key=viterbi_table[-1].get
    )

    # Backtrack
    best_tags = [best_final_tag]

    for i in range(len(words) - 1, 0, -1):

        previous_tag = backpointer[i][best_tags[-1]]

        if previous_tag is None:
            break

        best_tags.append(previous_tag)

    best_tags.reverse()

    return best_tags

In [12]:
sentence = "The student reads a book"

words = sentence.lower().split()

predicted_tags = viterbi(words)

for word, tag in zip(words, predicted_tags):

    print(f"{word} → {tag}")

the → DET
student → NOUN
reads → VERB
a → DET
book → NOUN


In [13]:
emission = emission_probs[
    current_tag
].get(word, 1e-6)